In [1]:
# Cell 1: Confirm Colab gave us a GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
# Cell 2: Install zstd (unpacking) and pciutils + lshw (GPU detection)
!apt-get install -y -q zstd pciutils lshw

Reading package lists...
Building dependency tree...
Reading state information...
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
pciutils is already the newest version (1:3.10.0-2build1).
lshw is already the newest version (02.19.git.2021.06.19.996aaad9c7-2ubuntu0.24.04.1).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.


In [3]:
# Cell 3: Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
# Cell 4: Start the Ollama server in the background
!nohup ollama serve > ollama.log 2>&1 &
!sleep 8

In [5]:
# Cell 5: Install the Ollama Python client
!pip install -q ollama

In [6]:
# Cell 6: Download the model through the Python client
import ollama

print(ollama.pull("gemma4:e2b").status)

success


In [7]:
# Cell 7: Check the server and the model
!ollama list

NAME                        ID              SIZE      MODIFIED               
gemma4:e2b                  b37049369adf    4.6 GB    Less than a second ago    
mxbai-embed-large:latest    468836162de7    669 MB    5 minutes ago             
nomic-embed-text:latest     0a109f422b47    274 MB    4 hours ago               


### Cell 8: Write tools.py to the session


In [8]:
%%writefile tools.py
"""Tools for interacting with the product catalog (synthetic data)."""

PRICES = {
    "mccb": 145.00,
    "acb": 2850.00,
    "busbar": 95.00,
    "current transformer": 38.00,
    "cable gland": 25.00,
}


def lookup_price(part_name: str) -> str:
    """Return the unit price of one part, as text (for example 95.0).
    Valid part names: mccb (moulded case circuit breaker),
    acb (air circuit breaker), busbar (copper busbar, per metre),
    current transformer, cable gland.
    Use the exact name in lowercase.
    If the name is not valid, the reply says so and lists the valid part names.
    """
    #Normalize the part name before look up
    name = part_name.strip().lower()
    if name in PRICES:
        return str(PRICES[name])
    return f"unknown part '{part_name}'. Valid part: {', '.join(PRICES)}"


def list_parts() -> str:
    """List every part in the catalog, separated by commas."""
    return ", ".join(PRICES)

def line_total(unit_price: float, quantity: int) -> float:
    """Return the total price for one quote line, rounded to 2 decimal places.
    unit_price is the price of one item in pounds. Use the value from lookup_price.
    quantity is the number of items, as a whole number.
    """
    # Round to 2 decimals so the amount is a clean currency value
    return round(unit_price * quantity, 2)

Writing tools.py


In [9]:
# Cell 9: Check that tools.py works on Colab
import tools

print(tools.lookup_price("busbar"))
print(tools.list_parts())
print(tools.line_total(95.0, 3))

95.0
mccb, acb, busbar, current transformer, cable gland
285.0


### Cell 10: Write schemas.py to the session


In [9]:
%%writefile schemas.py

from pydantic import BaseModel

class ToolStep(BaseModel):
    name: str
    arguments: dict
    result: str

class AgentResult(BaseModel):
    answer: str
    steps: list[ToolStep]

Writing schemas.py


### Cell 11: Write agent.py to the session


In [31]:
%%writefile agent.py

"""Runs the tools from tools.py safely and drives the agent loop."""

import ollama

from schemas import AgentResult, ToolStep
from tools import line_total, list_parts, lookup_price

MODEL = "gemma4:e2b"

TOOLS = {
    "lookup_price": lookup_price,
    "list_parts": list_parts,
    "line_total": line_total,
}


def run_tool(name: str, arguments: dict) -> str:
    """Run one tool by name and return its result as text.
    An unknown tool name or wrong inputs return a message instead of raising an error."""
    # Reject names the model made up
    if name not in TOOLS:
        return f"Unknown tool '{name}'. Valid tools: {', '.join(TOOLS)}"
    try:
        return str(TOOLS[name](**arguments))
    except TypeError as error:
        # Raised when the inputs do not match the tool's parameters
        return f"Bad inputs for '{name}': {error}"


def run_agent(question: str, max_rounds: int = 5) -> AgentResult:
    """Ask the model, run the tools it requests, and return the answer with the steps taken."""
    # The history starts with the user's question
    messages = [{"role": "user", "content": question}]
    # A record of every tool run, kept for the final result
    steps: list[ToolStep] = []

    for round_number in range(1, max_rounds + 1):
        print(f"Round {round_number}")

        # Ask the model, sending the whole history and the tool functions
        response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=list(TOOLS.values()),
            think=False,
        )

        # Keep the model's reply in the history
        messages.append(response.message)

        # No tool request means the model has finished
        if not response.message.tool_calls:
            return AgentResult(answer=response.message.content or "", steps=steps)

        # Run each requested tool, record the step, and keep the result in the history
        for call in response.message.tool_calls:
            name = call.function.name
            arguments = dict(call.function.arguments)
            result = run_tool(name, arguments)
            print(f"  {name}({arguments}) -> {result}")
            steps.append(ToolStep(name=name, arguments=arguments, result=result))
            messages.append({"role": "tool", "tool_name": name, "content": result})

    return AgentResult(answer="Stopped: reached the round limit.", steps=steps)

Overwriting agent.py


In [32]:
# Cell 12: Check that agent.py imports and its tool runner works
import agent

print(agent.run_tool("lookup_price", {"part_name": "busbar"}))
print(agent.run_tool("make_coffee", {}))
print(agent.run_agent("hello", max_rounds=0))

95.0
Unknown tool 'make_coffee'. Valid tools: lookup_price, list_parts, line_total
Stopped: reached the round limit.


In [19]:
# Cell 13: Run the agent on a question that needs no tool
from agent import run_agent

print(run_agent("What is the capital of France?"))

Round 1
The capital of France is Paris.


In [33]:
# Cell 14: Reload agent.py so the new version is used
import importlib

import agent

importlib.reload(agent)
print("reloaded")

reloaded


In [21]:
# Cell 15: Run the agent on a question that needs no tool
print(agent.run_agent("What is the capital of France?"))

Round 1
The capital of France is Paris.


In [22]:
# Cell 16: Run the agent on a question that needs one tool
print(agent.run_agent("What is the price of one busbar?"))

Round 1
  lookup_price({'part_name': 'busbar'}) -> 95.0
Round 2
The price of one busbar is 95.0.


In [23]:
# Cell 17: Run the agent on a question that needs two tools in a row
print(agent.run_agent("How much do 3 busbars cost in total?"))

Round 1
  lookup_price({'part_name': 'busbar'}) -> 95.0
Round 2
  line_total({'quantity': 3, 'unit_price': 95}) -> 285
Round 3
The total cost for 3 busbars is 285.00.


In [24]:
# Cell 18: Check that bad requests return messages instead of crashing
print(agent.run_tool("make_coffee", {}))
print(agent.run_tool("lookup_price", {"part_name": "fuse"}))
print(agent.run_tool("line_total", {"unit_price": 95.0}))

Unknown tool 'make_coffee'. Valid tools: lookup_price, list_parts, line_total
unknown part 'fuse'. Valid part: mccb, acb, busbar, current transformer, cable gland
Bad inputs for 'line_total': line_total() missing 1 required positional argument: 'quantity'


In [25]:
# Cell 19: Run the agent on a question that needs the parts list
print(agent.run_agent("Which parts can you price?"))

Round 1
I can price the following parts: mccb (moulded case circuit breaker), acb (air circuit breaker), busbar (copper busbar, per metre), current transformer, and cable gland.


In [26]:
# Cell 20: Ask the agent to use the list_parts tool explicitly
print(agent.run_agent("Call the list_parts tool and show me what it returns."))

Round 1
  list_parts({}) -> mccb, acb, busbar, current transformer, cable gland
Round 2



In [27]:
# Cell 21: Show the exact return value, including an empty reply
answer = agent.run_agent("Call the list_parts tool and show me what it returns.")
print(repr(answer))

Round 1
  list_parts({}) -> mccb, acb, busbar, current transformer, cable gland
Round 2
''


In [34]:
# Cell 22: Load the new version
import sys

sys.modules.pop("agent", None)
import agent

print("loaded")

loaded


In [35]:
# Cell 23: Run the two-tool question and read the structured result
result = agent.run_agent("How much do 3 busbars cost in total?")
print(result.answer)
for step in result.steps:
    print(step.name, step.arguments, "->", step.result)

Round 1
  lookup_price({'part_name': 'busbar'}) -> 95.0
Round 2
  line_total({'quantity': 3, 'unit_price': 95}) -> 285
Round 3
The total cost for 3 busbars is 285.00 pounds.
lookup_price {'part_name': 'busbar'} -> 95.0
line_total {'quantity': 3, 'unit_price': 95} -> 285
